In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip -q install ijson pycocotools pyyaml

In [ ]:
import os
for p in ["/content/drive/MyDrive/M18K_Reproduction",
          "/content/drive/MyDrive/Colab Notebooks/M18K_Reproduction"]:
    print(p, "| tồn tại:", os.path.exists(p))
    if os.path.exists(p): print("   ", os.listdir(p))

In [ ]:
from google.colab import drive
drive.mount("/content/drive", force_remount=True)

import os
B = "/content/drive/MyDrive/Colab Notebooks/M18K_Reproduction"
print(os.listdir(B))

In [ ]:
from google.colab import drive
drive.mount("/content/drive", force_remount=True)

import os
BASE = "/content/drive/MyDrive/Colab Notebooks/M18K_Reproduction"
D = f"{BASE}/M18KV2"
print(os.listdir(BASE))
print("tồn tại:", os.path.exists(D))
if os.path.exists(D):
    print(os.listdir(D))
    for s in ["train", "valid", "test"]:
        p = f"{D}/{s}"
        print(s, "| rgb:", os.path.exists(f"{p}/rgb"), "| json:", os.path.exists(f"{p}/annotations_coco.json"))

In [ ]:
!pip -q install ijson pycocotools pyyaml

BASE = "/content/drive/MyDrive/Colab Notebooks/M18K_Reproduction"
R = f"{BASE}/wb_task_repo"
D = f"{BASE}/M18KV2"

!ls -R "{R}" | head -20
!python "{R}/07_source_code/data/build_wb_dataset.py" --mode build --drive-root "{BASE}" --data-root "{D}"

In [ ]:
import json, sys
sys.path.insert(0, f"{R}/07_source_code/data")
import build_wb_dataset as b

S = json.load(open(f"{R}/09_results/metrics/wb_conversion_summary.json"))
print("skipped reasons:", {sp: d["skipped"] for sp, d in S["splits"].items()})
print("category mapping:", S["source_category_mapping"]["train"])

cs = b.CocoStream(f"{D}/train/annotations_coco.json")
roles = b.map_categories(list(cs.items("categories")))
imgs = {i["id"]: i for i in cs.items("images")}
for a in cs.items("annotations"):
    if roles.get(a["category_id"]) != "WB": continue
    im = imgs[a["image_id"]]
    st = b.new_stats()
    if b.convert_wb(a, im["width"], im["height"], st) is None:
        seg = a.get("segmentation")
        print("ann_id", a["id"], "image", im["file_name"], "| reason", dict(st["skipped"]),
              "| seg type", type(seg).__name__, "| n_rings", len(seg) if seg else 0,
              "| ring lens", [len(r) for r in seg] if isinstance(seg, list) else None, "| area", a.get("area"))

In [ ]:
!python "{R}/07_source_code/evaluation/mask_area.py" --coco-json "{D}/train/annotations_coco.json" --image-dir "{D}/train/rgb" --out-csv "{R}/09_results/metrics/wb_area_sanity.csv"

In [ ]:
!python "{R}/07_source_code/data/build_wb_dataset.py" --mode report --drive-root "{BASE}" --data-root "{D}"
!cat "{R}/09_results/reports/wb_dataset_measurement.md"

In [ ]:
!pip -q install ijson pycocotools pyyaml
!python "{R}/07_source_code/data/build_wb_dataset.py" --mode rebuild --drive-root "{BASE}" --data-root "{D}"

In [ ]:
!python "{R}/07_source_code/data/build_wb_dataset.py" --mode report --drive-root "{BASE}" --data-root "{D}"

import json
S = json.load(open(f"{R}/09_results/metrics/wb_conversion_summary.json"))
for sp, d in S["splits"].items():
    print(sp, d["ann_by_role"], d["ann_by_category_id"], "non_wb_omitted:", d["non_wb_omitted"])

extra = """
## Additional notes (from executed scripts)
- Source categories: 0 `Mushrooms` (omitted), 1 `BB` (omitted), 2 `WB` (kept, remapped to YOLO class 0).
- Train: 6 WB annotations skipped (ann_id 65633, 68399, 151483, 398470, 528181, 543586). Each is a single 12-point polygon with COCO `area = 0.0`. Whether they were already degenerate before clipping to the image is NOT_VERIFIED; the script labels the reason `degenerate_after_clip`.
- No `iscrowd` WB annotations and no multi-component WB polygons were found in any split, so largest-component-only discarded nothing on this dataset.
- Area test: pycocotools rows have expected = RLE area of the same polygon, so they validate the pixel-counting function, not the rasterisation rule. The cv2.fillPoly cross-check differs by 133-169 px2 (rasterisation-rule difference).
- Fresh-runtime rebuild: `--mode rebuild` returned REBUILD OK (see 09_results/metrics/wb_rebuild_verification.json).
"""
open(f"{R}/09_results/reports/wb_dataset_measurement.md", "a").write(extra)
!cat "{R}/09_results/reports/wb_dataset_measurement.md"

In [ ]:
   import pandas as pd
   pd.set_option("display.max_colwidth", None)
   print(pd.read_csv(f"{R}/09_results/metrics/wb_area_sanity.csv")[["case", "source"]].tail(6))

In [ ]:
p = f"{R}/09_results/reports/wb_dataset_measurement.md"
t = open(p).read()
t = t.split("## Additional notes")[0]
t = t.replace(
 "- Largest-component-only labels slightly understate multi-part instances (see losses above); area measurement from COCO polygons uses all components.",
 "- Largest-component-only is the conversion rule for multi-part polygons, but no multi-component WB polygon exists in any split, so nothing was discarded; area measurement from COCO polygons uses all components.")
t += """## Additional notes (from executed scripts)
- COCO categories: 0 `Mushrooms` (declared, 0 annotations in all splits), 1 `BB` (omitted: 686 train / 75 val / 97 test), 2 `WB` (kept, remapped to YOLO class 0).
- Train: 6 WB annotations skipped (ann_id 65633, 68399, 151483, 398470, 528181, 543586). Each is a single 12-point polygon with COCO `area = 0.0`. The build summary shows 0 annotations clipped to the image in train, so these polygons were degenerate before clipping; the script labels the reason `degenerate_after_clip`. Their images still contain other WB instances (0 empty label files).
- No `iscrowd` WB annotations and no multi-component WB polygons in any split; largest-component-only discarded nothing.
- Area test: the 3 real masks are COCO ann_id 0, 1, 2, all from image_id 0 (1280x720; file size checked equal to COCO size for that image only). pycocotools rows use the RLE area of the same polygon as expected, so they validate the pixel-counting function, not the rasterisation rule. The cv2.fillPoly cross-check differs by 133-169 px2 (rasterisation-rule difference).
- Fresh-runtime rebuild: `--mode rebuild` returned REBUILD OK (09_results/metrics/wb_rebuild_verification.json).
"""
open(p, "w").write(t)

In [ ]:
import zipfile
z = zipfile.ZipFile(f"{BASE}/derived_datasets/wb_yolo_labels.zip")
names = z.namelist()
print(len(names), [n for n in names if not n.startswith("labels/")])
print("file ảnh trong zip:", [n for n in names if n.lower().endswith((".png", ".jpg", ".jpeg"))])

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
BASE = "/content/drive/MyDrive/Colab Notebooks/M18K_Reproduction"
R = f"{BASE}/wb_task_repo"
D = f"{BASE}/M18KV2"
!pip -q install ijson pycocotools pyyaml
!python "{R}/07_source_code/data/build_wb_dataset.py" --mode rebuild --drive-root "{BASE}" --data-root "{D}"

03/11


In [ ]:
from google.colab import drive
drive.mount("/content/drive")
!pip -q install ultralytics gdown pycocotools

import os, gdown
BASE = "/content/drive/MyDrive/Colab Notebooks/M18K_Reproduction"
R = f"{BASE}/wb_task_repo"
D = f"{BASE}/M18KV2"
CK = "/content/wb_yolov8n_seg.pt"
if not os.path.exists(CK):
    gdown.download(id="1Q5iq_-j-RwOTkyiiXyTOc80Cf_xwk7K-", output=CK, quiet=False)
print("checkpoint:", os.path.exists(CK), os.path.getsize(CK) if os.path.exists(CK) else "")
print("test rgb:", os.path.exists(f"{D}/test/rgb"), "| test json:", os.path.exists(f"{D}/test/annotations_coco.json"))

In [ ]:
!python "{R}/07_source_code/evaluation/area_evaluation.py" --data-root "{D}" --repo-root "{R}" --checkpoint "{CK}"

In [ ]:
import pandas as pd, numpy as np, torch
O = f"{R}/09_results/area_evaluation"
m = pd.read_csv(f"{O}/mask_area_matches.csv")

print("rel error pct p50/p75/p90/p95/p99/max:", np.percentile(m.relative_error_pct, [50,75,90,95,99]).round(2), round(float(m.relative_error_pct.max()), 2))
d = m.pred_area_px2 - m.gt_area_px2
print("pred larger than GT: %.1f%% | smaller: %.1f%% | equal: %.1f%%" % (100*(d>0).mean(), 100*(d<0).mean(), 100*(d==0).mean()))
print("mask IoU of matched: mean %.3f, min %.3f" % (m.mask_iou.mean(), m.mask_iou.min()))

m["size_bin"] = pd.qcut(m.gt_area_px2, 3, labels=["small","medium","large"])
print(m.groupby("size_bin", observed=True).agg(n=("abs_error_px2","size"), gt_mean=("gt_area_px2","mean"),
      mae=("abs_error_px2","mean"), rel=("relative_error_pct","mean"), signed=("abs_error_px2","size")).drop(columns="signed").round(1))

up = pd.read_csv(f"{O}/unmatched_predictions.csv"); ug = pd.read_csv(f"{O}/unmatched_gt.csv")
print("\nunmatched preds:", len(up), "| area median %.0f | conf median %.2f | best IoU with any GT <0.1: %d, 0.1-0.5: %d" %
      (up.pred_area_px2.median(), up.pred_conf.median(), (up.best_iou_with_any_gt<0.1).sum(),
       ((up.best_iou_with_any_gt>=0.1)&(up.best_iou_with_any_gt<0.5)).sum()))
print("unmatched GT:", len(ug), "| area median %.0f | best IoU with any pred <0.1: %d, 0.1-0.5: %d" %
      (ug.gt_area_px2.median(), (ug.best_iou_with_any_pred<0.1).sum(),
       ((ug.best_iou_with_any_pred>=0.1)&(ug.best_iou_with_any_pred<0.5)).sum()))

pi = pd.read_csv(f"{O}/per_image_summary.csv")
r = pi.pred_total_area_px2 / pi.gt_total_area_px2
print("\nper-image pred/GT total area ratio: mean %.3f, min %.3f, max %.3f" % (r.mean(), r.min(), r.max()))
print("GPU available in this runtime:", torch.cuda.is_available())

In [ ]:
from IPython.display import display, Image
import glob, os

O = f"{R}/09_results/area_evaluation/area_examples"
for p in sorted(glob.glob(f"{O}/*.png")):
    print(os.path.basename(p))
    display(Image(filename=p, width=1200))

In [ ]:
import pandas as pd
m = pd.read_csv(f"{R}/09_results/area_evaluation/mask_area_matches.csv")
m["signed"] = m.pred_area_px2 - m.gt_area_px2
m["bin"] = pd.qcut(m.gt_area_px2, [0, .33, .66, .9, .99, 1], labels=["nhỏ", "vừa", "lớn", "rất lớn(top10%)", "top1%"])
print(m.groupby("bin", observed=True).agg(n=("signed", "size"), gt_mean=("gt_area_px2", "mean"),
      signed_mean=("signed", "mean"), pct_pred_smaller=("signed", lambda s: 100 * (s < 0).mean())).round(1))